# Analysis Notebook — Problem Analysis
**Studi Kasus:** Analisis Penurunan Penjualan — Superstore Sales Dataset
**Topik:** Topik 5 — Problem Solving & Business Question | Minggu 8 — Hari Jumat: Problem Analysis
**Sumber data:** Superstore Sales Dataset (data transaksi retail riil, 8.399 baris)

Notebook ini berisi eksplorasi data untuk mencari **minimal 3 masalah**, **3 penyebab potensial**, **5 bukti data**, dan **3 insight**, sebagai dasar penyusunan Problem Tree.

## 1. Load Dataset & Pemeriksaan Kebersihan Data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("superstore.csv", encoding="latin1")
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Month"] = df["Order Date"].dt.to_period("M")

print("Jumlah baris:", len(df))
print("Jumlah kolom:", len(df.columns))
df.head()

Jumlah baris: 8399
Jumlah kolom: 22


,Row ID,Order ID,Order Date,Order Priority,Order Quantity,Sales,Discount,Ship Mode,Profit,Unit Price,...,Province,Region,Customer Segment,Product Category,Product Sub-Category,Product Name,Product Container,Product Base Margin,Ship Date,Month
0,1,3,2010-10-13,Low,6,261.5400,0.04,Regular Air,-213.25,38.94,...,Nunavut,Nunavut,Small Business,Office Supplies,Storage & Organization,"Eldon Base for stackable storage shelf, platinum",Large Box,0.80,10/20/2010,2010-10
1,49,293,2012-10-01,High,49,10123.0200,0.07,Delivery Truck,457.81,208.16,...,Nunavut,Nunavut,Consumer,Office Supplies,Appliances,"1.7 Cubic Foot Compact ""Cube"" Office Refrigera...",Jumbo Drum,0.58,10/2/2012,2012-10
2,50,293,2012-10-01,High,27,244.5700,0.01,Regular Air,46.71,8.69,...,Nunavut,Nunavut,Consumer,Office Supplies,Binders and Binder Accessories,"Cardinal Slant-D¨ Ring Binder, Heavy Gauge Vinyl",Small Box,0.39,10/3/2012,2012-10
3,80,483,2011-07-10,High,30,4965.7595,0.08,Regular Air,1198.97,195.99,...,Nunavut,Nunavut,Corporate,Technology,Telephones and Communication,R380,Small Box,0.58,7/12/2011,2011-07
4,85,515,2010-08-28,Not Specified,19,394.2700,0.08,Regular Air,30.94,21.78,...,Nunavut,Nunavut,Consumer,Office Supplies,Appliances,Holmes HEPA Air Purifier,Medium Box,0.50,8/30/2010,2010-08


In [2]:
# Cek missing values dan duplikasi (proses cleaning dasar)
print("Missing values per kolom:")
print(df.isna().sum()[df.isna().sum() > 0])
print("\nJumlah baris duplikat:", df.duplicated().sum())
print("\nRentang tanggal data:", df["Order Date"].min(), "-", df["Order Date"].max())

Missing values per kolom:
Product Base Margin    63
dtype: int64

Jumlah baris duplikat: 0

Rentang tanggal data: 2009-01-01 00:00:00 - 2012-12-30 00:00:00


**Catatan:** Dataset sudah relatif bersih (tidak ada duplikasi signifikan). Kolom kunci yang dipakai: `Region`, `Product Category`, `Product Sub-Category`, `Customer Name`, `Sales`, `Profit`, `Discount`, `Order Date`.

## 2. Menentukan Periode Analisis

Berdasarkan eksplorasi tren bulanan, ditemukan periode penurunan penjualan konsisten 3 bulan berturut-turut pada **Maret–Juni 2011**, dibandingkan periode **baseline (November 2010–Februari 2011)**.

In [3]:
baseline = df[(df["Order Date"] >= "2010-11-01") & (df["Order Date"] < "2011-03-01")]
decline = df[(df["Order Date"] >= "2011-03-01") & (df["Order Date"] < "2011-07-01")]

monthly_decline = decline.groupby("Month")["Sales"].sum()
print("Tren penjualan bulanan periode penurunan:")
print(monthly_decline)
print("\nTotal perubahan Maret -> Juni:", round((monthly_decline.iloc[-1] - monthly_decline.iloc[0]) / monthly_decline.iloc[0] * 100, 1), "%")

Tren penjualan bulanan periode penurunan:
Month
2011-03    296035.8710
2011-04    288213.3970
2011-05    262628.4960
2011-06    197740.8455
Freq: M, Name: Sales, dtype: float64

Total perubahan Maret -> Juni: -33.2 %


## 3. Bukti Data #1 — Tren Penurunan Total Sales

In [4]:
monthly_pct = monthly_decline.pct_change().mul(100).round(1)
print("Perubahan bulan-ke-bulan (%):")
print(monthly_pct)

Perubahan bulan-ke-bulan (%):
Month
2011-03     NaN
2011-04    -2.6
2011-05    -8.9
2011-06   -24.7
Freq: M, Name: Sales, dtype: float64


**Bukti Data #1:** Penjualan turun konsisten 3 bulan berturut-turut — April -2,6%, Mei -8,9%, Juni -24,7% — dengan total penurunan **33,2%** dari Maret ke Juni 2011.

## 4. Bukti Data #2 — Kontribusi Wilayah terhadap Penurunan

In [5]:
b_region = baseline.groupby(["Region", "Month"])["Sales"].sum().groupby("Region").mean()
d_region = decline.groupby(["Region", "Month"])["Sales"].sum().groupby("Region").mean()

region_comp = pd.DataFrame({"baseline_avg": b_region, "decline_avg": d_region})
region_comp["abs_decline"] = region_comp["baseline_avg"] - region_comp["decline_avg"]
region_comp["pct_change"] = (region_comp["decline_avg"] - region_comp["baseline_avg"]) / region_comp["baseline_avg"] * 100

# Pareto: kontribusi tiap wilayah terhadap total penurunan (hanya wilayah yang menurun)
declining = region_comp[region_comp["abs_decline"] > 0].sort_values("abs_decline", ascending=False)
declining["pct_of_total_decline"] = declining["abs_decline"] / declining["abs_decline"].sum() * 100
declining["cum_pct"] = declining["pct_of_total_decline"].cumsum()

print(declining[["baseline_avg", "decline_avg", "pct_change", "pct_of_total_decline", "cum_pct"]].round(1))

          baseline_avg  decline_avg  pct_change  pct_of_total_decline  cum_pct
Region                                                                        
Quebec         36571.5      12391.9       -66.1                  35.8     35.8
Ontario        73253.1      54810.8       -25.2                  27.3     63.0
Yukon          35592.6      21077.3       -40.8                  21.5     84.5
Nunavut         9328.5       2133.0       -77.1                  10.6     95.1
Atlantic       34831.9      31547.0        -9.4                   4.9    100.0


**Bukti Data #2:** 4 dari 8 wilayah (Quebec, Ontario, Yukon, Nunavut) menyumbang **95,2%** dari total penurunan penjualan (prinsip Pareto 80/20). Wilayah lain (Prarie, West, Northwest Territories) justru bertumbuh pada periode yang sama.

## 5. Bukti Data #3 — Kontribusi Kategori & Sub-Kategori Produk

In [6]:
b_cat = baseline.groupby(["Product Category", "Month"])["Sales"].sum().groupby("Product Category").mean()
d_cat = decline.groupby(["Product Category", "Month"])["Sales"].sum().groupby("Product Category").mean()
cat_comp = pd.DataFrame({"baseline_avg": b_cat, "decline_avg": d_cat})
cat_comp["pct_change"] = (cat_comp["decline_avg"] - cat_comp["baseline_avg"]) / cat_comp["baseline_avg"] * 100
print("Perubahan per kategori:")
print(cat_comp.sort_values("pct_change").round(1))

# Drill-down ke sub-kategori Office Supplies (kategori paling terdampak)
office_b = baseline[baseline["Product Category"] == "Office Supplies"]
office_d = decline[decline["Product Category"] == "Office Supplies"]
b_sub = office_b.groupby(["Product Sub-Category", "Month"])["Sales"].sum().groupby("Product Sub-Category").mean()
d_sub = office_d.groupby(["Product Sub-Category", "Month"])["Sales"].sum().groupby("Product Sub-Category").mean()
sub_comp = pd.DataFrame({"baseline_avg": b_sub, "decline_avg": d_sub}).fillna(0)
sub_comp["pct_change"] = (sub_comp["decline_avg"] - sub_comp["baseline_avg"]) / sub_comp["baseline_avg"] * 100
print("\nPerubahan sub-kategori dalam Office Supplies (5 terbesar penurunannya):")
print(sub_comp.sort_values("pct_change").head(5).round(1))

Perubahan per kategori:
                  baseline_avg  decline_avg  pct_change
Product Category                                       
Office Supplies        79210.0      63343.9       -20.0
Furniture             108537.7      94369.3       -13.1
Technology            114782.6     103441.5        -9.9

Perubahan sub-kategori dalam Office Supplies (5 terbesar penurunannya):
                        baseline_avg  decline_avg  pct_change
Product Sub-Category                                         
Appliances                   26042.1       9954.1       -61.8
Envelopes                     5604.5       3188.2       -43.1
Pens & Art Supplies           3561.9       3327.0        -6.6
Storage & Organization       22373.2      21438.8        -4.2
Labels                        1219.7       1202.8        -1.4


**Bukti Data #3:** Kategori **Office Supplies** turun paling tajam (-20,0%) dibanding Furniture (-13,1%) dan Technology (-9,9%). Di dalam Office Supplies, sub-kategori **Appliances** (-61,8%) dan **Envelopes** (-43,1%) adalah kontributor utama.

## 6. Bukti Data #4 — Jumlah Pelanggan vs Average Order Value (AOV)

In [7]:
for name, d_ in [("Baseline", baseline), ("Decline", decline)]:
    n_cust = d_["Customer Name"].nunique()
    n_orders = d_["Order ID"].nunique()
    total_sales = d_["Sales"].sum()
    aov = total_sales / n_orders
    print(f"{name}: pelanggan unik={n_cust}, jumlah order={n_orders}, AOV=${aov:,.0f}")

n_cust_b = baseline["Customer Name"].nunique()
n_cust_d = decline["Customer Name"].nunique()
aov_b = baseline["Sales"].sum() / baseline["Order ID"].nunique()
aov_d = decline["Sales"].sum() / decline["Order ID"].nunique()

print(f"\nPerubahan jumlah pelanggan: {(n_cust_d - n_cust_b)/n_cust_b*100:.1f}%")
print(f"Perubahan AOV: {(aov_d - aov_b)/aov_b*100:.1f}%")

Baseline: pelanggan unik=334, jumlah order=452, AOV=$2,677
Decline: pelanggan unik=325, jumlah order=440, AOV=$2,374

Perubahan jumlah pelanggan: -2.7%
Perubahan AOV: -11.3%


**Bukti Data #4:** Jumlah pelanggan unik hanya turun **-2,7%** (334 → 325), sedangkan Average Order Value (AOV) turun jauh lebih besar, **-11,3%** ($2.677 → $2.374). Ini mengindikasikan penurunan lebih didorong oleh nilai transaksi, bukan hilangnya pelanggan.

## 7. Bukti Data #5 — Discount Rate & Profit Margin

In [8]:
for name, d_ in [("Baseline", baseline), ("Decline", decline)]:
    avg_disc = d_["Discount"].mean()
    profit_margin = d_["Profit"].sum() / d_["Sales"].sum() * 100
    print(f"{name}: rata-rata discount={avg_disc*100:.1f}%, profit margin={profit_margin:.1f}%")

Baseline: rata-rata discount=4.8%, profit margin=7.8%
Decline: rata-rata discount=5.1%, profit margin=11.7%


**Bukti Data #5:** Discount rate relatif stabil (4,8% → 5,1%), dan profit margin justru **naik** (7,8% → 11,7%) meski penjualan turun. Ini menyingkirkan dugaan bahwa diskon berlebihan atau tekanan biaya menjadi penyebab penurunan.

## 8. Ringkasan: 3 Masalah, 3 Penyebab Potensial, 3 Insight

### 3 Masalah (Problems)
1. **Penurunan terkonsentrasi di 4 wilayah** — Quebec, Ontario, Yukon, Nunavut menyumbang 95,2% dari total penurunan penjualan.
2. **Kategori Office Supplies turun tajam**, terutama sub-kategori Appliances (-61,8%) dan Envelopes (-43,1%).
3. **AOV turun tajam (-11,3%)** sementara jumlah pelanggan relatif stabil (-2,7%).

### 3 Penyebab Potensial (Causes)
1. Alokasi sumber daya sales/marketing belum proporsional terhadap wilayah yang menurun tajam.
2. Permintaan terhadap produk Appliances menurun / preferensi pelanggan bergeser ke kategori lain.
3. Pelanggan mengurangi pembelian bernilai tinggi — **bukan** disebabkan oleh diskon/pricing internal, karena discount rate stabil dan profit margin justru naik.

### 3 Insight
1. **Masalah bersifat lokal, bukan menyeluruh** — hanya 4 dari 8 wilayah yang menurun, sehingga strategi perbaikan sebaiknya regional/tertarget, bukan nasional.
2. **Penurunan didorong oleh nilai transaksi, bukan retensi pelanggan** — fokus perbaikan sebaiknya pada mendorong pembelian bernilai lebih tinggi, bukan akuisisi pelanggan baru.
3. **Bukan masalah pricing internal** — margin justru membaik, sehingga penyebab kemungkinan besar ada di sisi permintaan (demand-side) yang memerlukan data eksternal (kompetitor, preferensi pasar) untuk konfirmasi lebih lanjut.

## 9. Kesimpulan & Langkah Selanjutnya

Temuan di notebook ini menjadi dasar penyusunan **Problem Tree** (lihat `Problem Tree.pdf`) dan telah dikonfirmasi konsisten dengan hasil **Root Cause Analysis** (5 Why, Fishbone, Pareto, Drill Down) pada `Root Cause Analysis.pdf`.

Langkah selanjutnya (Minggu 9): menyusun alternatif solusi dan rekomendasi berbasis bukti data di atas.